# Research on correlation of books rating and its commercial popularity                    

### Step 1: Loading the Data

In [36]:
import pandas as pd
import altair as alt

# Load Top 500 Novels dataset (tab-separated)
novels_df = pd.read_csv(
    "https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/main/datasets/top-500-novels/final_merged_dataset_no_full_text.tsv",
    sep="\t"
)

# Load NYT Bestsellers dataset
nyt_df = pd.read_csv(
    "https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv",
    sep="\t",
    encoding="utf-8"
)
# Shape
print("Top 500 Novels shape:", novels_df.shape)
print("NYT Bestsellers shape:", nyt_df.shape)

Top 500 Novels shape: (500, 29)
NYT Bestsellers shape: (60386, 6)


### Step 2: Aggragate NYT dataset: show how often books were on the top list and what was the highest ranking they got

In [37]:
nyt_per_book = (
    nyt_df.groupby(["title_id", "title", "author"])
    .agg(
        weeks_on_list=("week", "count"),
        best_rank=("rank","min"),
    )
    .reset_index()
)

print("Shape (one row per book):", nyt_per_book.shape)                      
nyt_per_book.sort_values("weeks_on_list", ascending=False)

Shape (one row per book): (7427, 5)


,title_id,title,author,weeks_on_list,best_rank
3339,3343,"OH, THE PLACES YOU'LL GO!",Dr. Seuss,178,1
4914,4918,THE DA VINCI CODE,Dan Brown,165,1
4792,4796,THE CELESTINE PROPHECY,James Redfield,165,1
4719,4723,THE BRIDGES OF MADISON COUNTY,Robert James Waller,164,1
380,381,ALL THE LIGHT WE CANNOT SEE,Anthony Doerr,132,1
...,...,...,...,...,...
6088,6092,THE RACE,Richard North Patterson,1,15
4507,4511,THE A LIST,J.A. Jance,1,13
875,876,BURN,Linda Howard,1,6
876,877,BURN,Nevada Barr,1,12


Step 3: Match NYT dataset + Novels dataset on author and name of the book. Caclulate the average rating for each book from Goodreads

In [38]:
novels_df["title_key"] = novels_df["title"].str.lower().str.strip()
novels_df["author_key"]=novels_df["author"].str.lower().str.strip()

nyt_per_book["title_key"]=nyt_per_book["title"].str.lower().str.strip()
nyt_per_book["author_key"]=nyt_per_book["author"].str.lower().str.strip()

# Let's keep books that appear in both datasets only

merged_df = novels_df.merge(
    nyt_per_book[["title_key", "author_key", "weeks_on_list", "best_rank"]], 
    on=["title_key", "author_key"],
    how="inner",
)
print("Books in the both datasets:", merged_df.shape[0])
merged_df[["title", "author", "gr_avg_rating", "weeks_on_list", "best_rank"]].head(10)

Books in the both datasets: 126


,title,author,gr_avg_rating,weeks_on_list,best_rank
0,Brave New World,Aldous Huxley,3.99,2,3
1,To Kill a Mockingbird,Harper Lee,4.26,98,2
2,Animal Farm,George Orwell,3.99,6,7
3,The Grapes of Wrath,John Steinbeck,4.01,35,1
4,Ulysses,James Joyce,3.75,9,2
5,The Old Man and the Sea,Ernest Hemingway,3.80,26,3
6,The Catcher in the Rye,J.D. Salinger,3.80,29,4
7,Gone with the Wind,Margaret Mitchell,4.31,78,1
8,Of Mice and Men,John Steinbeck,3.88,16,4
9,Nineteen Eighty-Four,George Orwell,4.19,20,3


Step 4: Checking the correlation

I will use Pearson correlation coeffieicent between goodreads rating and 2 NYT popularity measurements:
1. "Weeks_on_list" - how long the book was in the list?
2."best_rank" - what's the highest position book ever reached?

Note: a *negative* correlation with `best_rank` would actually mean higher-rated books climb higher (since rank 1 is the best position).  

In [39]:
  corr_weeks = merged_df["gr_avg_rating"].corr(merged_df["weeks_on_list"])    
  corr_rank  = merged_df["gr_avg_rating"].corr(merged_df["best_rank"])
                                              
  print(f"Correlation: Goodreads rating vs. weeks on list: {corr_weeks:.3f}")
  print(f"Correlation: Goodreads rating vs. best rank:     {corr_rank:.3f}")  
                                                                              
  merged_df[["gr_avg_rating", "weeks_on_list", "best_rank"]].corr()

Correlation: Goodreads rating vs. weeks on list: 0.148
Correlation: Goodreads rating vs. best rank:     -0.053


,gr_avg_rating,weeks_on_list,best_rank
gr_avg_rating,1.000000,0.148485,-0.053122
weeks_on_list,0.148485,1.000000,-0.397424
best_rank,-0.053122,-0.397424,1.000000


Step 5: Visualize the results of coefficients with Altair

For this step, I will use scatter plot with regression line to show the relationship between the rating on Goodreads and weeks on the NYT list. X-axis shows Goodreads rating and y-axis shows weeks on the NYT bestseller database. The red regression line shows the trend.

Scatter Graph 1: Ratings vs Weeks Trending

In [40]:
scatter = alt.Chart(merged_df).mark_circle(size=80, opacity=0.6).encode(    
      x=alt.X("gr_avg_rating:Q",              
              title="Goodreads Average Rating",                               
              scale=alt.Scale(zero=False)),                       
      y=alt.Y("weeks_on_list:Q",                                              
              title="Weeks on NYT Bestseller List"),                          
      tooltip=["title", "author", "gr_avg_rating", "weeks_on_list",           
  "best_rank"],                                                               
  ).properties(                                                   
      width=600,                                                              
      height=400,                                                 
      title="Goodreads Rating vs. NYT Bestseller Longevity (n = 126 books)"   
  )                                                               
                                                                              
regression = scatter.transform_regression(                                  
      "gr_avg_rating", "weeks_on_list"                            
  ).mark_line(color="red")   
scatter + regression

alt.LayerChart(...)

Scatter Graph 2: Ratings vs Weeks Trending

In [41]:
scatter2 = alt.Chart(merged_df).mark_circle(size=80, opacity=0.6,           
  color="#4C78A8").encode(                                                    
      x=alt.X("gr_avg_rating:Q",          
              title="Goodreads Average Rating",                               
              scale=alt.Scale(zero=False)),                                   
      y=alt.Y("best_rank:Q",                                                  
              title="Best NYT Rank Achieved (1 = best)",                      
              scale=alt.Scale(reverse=True)), 
      tooltip=["title", "author", "gr_avg_rating", "best_rank"],              
  ).properties(                           
      width=600, height=400,                                                  
      title="Goodreads Rating vs. Peak NYT Position"                          
  )                                                                           
                                                                              
regression2 = scatter2.transform_regression(                    
      "gr_avg_rating", "best_rank"                                            
  ).mark_line(color="red")                    
                                                                              
scatter2 + regression2  

alt.LayerChart(...)

 ### Step 6: Findings & Interpretation   

  ### Headline result                                                         
  The Pearson correlation between Goodreads average rating and NYT weeks-on-list across the 126 overlapping books is **0.148**, and the correlation with best NYT rank is **-0.053**.  

   ### What this means

   1. If correlation is low (between -0.2 to +0.2): Goodreads ratings and NYT bestsellers appear to be not correlated, so-called independent. A book that is well read by readers doesnt predict commerical "long life", and book that stays a bestseller for a long time doesnt predict that it will be well-rated.
   2. If correlation is positive and moderate (+0.2 - +0.5): Higher-rated books tends to last long on the bestseller list
   3. If correlation is negative: higher-rated books spent less weeks on the top list - possible because cononical "literary" work that have high user rating, but aren't on the top lists for commercial success.
   

### Limitations of the analysis
1. The merged dataset is only **126 books** out of 500 since a lot o books in top  500 dataset was published before 1931, or not in english
2. For matching 2 datasets on 'author' and 'book' title I used lowercasing cleaning to standartaze them. I'm wondering if I might miss something that wasn't applied in both styles
3. Goodreads rating come from tech-savy people and younger population which might be biased based on the diversity of age and tech experience from all people.

### Future Research Directions

1. Look on the author overlap name instead on book name. The main reason - it would be interested to see how many authors have several books in the top of the list.

2. Check other correlations and check the work on the validity. Maybe, check for other, more suitable correlation methods.